[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/11_agents_tools_mcp/42_multi_agent_systems.ipynb)

# 📓 Notebook 42 — Multi-Agent Systems & Capstone

> **Module:** Agents, Tools & MCP · **Estimated time:** 80–100 min · **Difficulty:** Advanced

One agent with good tools solves most problems. But some tasks have **distinct sub-jobs** — analyse, then draft, then check — that are cleaner, more reliable, and more testable when handled by **specialist agents** under an **orchestrator**. This final notebook covers the multi-agent patterns that matter, their failure modes, and a **capstone** that combines everything in the module: agents (NB 39), robust tools (NB 40), and an MCP server (NB 41) into a working **support-operations assistant**.

100% offline. The deterministic stand-ins mark exactly where a real LLM plugs in.

## 🎯 Learning objectives

By the end you can:

1. Decide **when multiple agents help — and when they're overkill**.
2. Build the **orchestrator–worker** pattern with specialist agents.
3. Implement **routing** and **handoff** between agents.
4. Share state safely via a **blackboard**.
5. **Evaluate** a multi-agent system end-to-end.
6. Assemble a **capstone**: orchestrator + specialists + tools + an MCP server.

## ✅ Prerequisites

Notebooks 39 (agents), 40 (tools), 41 (MCP). This notebook reuses their ideas; the code here is self-contained.

## 1. When to use more than one agent

Multi-agent is a **cost** (more calls, more latency, more ways to fail), so spend it only when it buys something:

| Use multiple agents when… | Use **one** agent when… |
|---|---|
| sub-tasks need **different tools / instructions** | one toolset and prompt covers it |
| you want **independent review** (writer + critic) | the task is a single straight line |
| sub-tasks run **in parallel** | steps are strictly sequential & cheap |
| roles map to **different owners/permissions** | everything shares one permission |

> Rule of thumb: start with one agent. Split only when a single prompt is doing two jobs badly.

## 2. A specialist agent

A **specialist** is just an agent with a focused role and a *subset* of tools. Narrow scope = better reliability and easier testing. Offline, each specialist's "brain" is a small deterministic `act()`; a real one calls an LLM with its role as the system prompt.

In [ ]:
import json, re
from dataclasses import dataclass, field
from typing import Callable

# Shared toolbox (in NB 41 these came from an MCP server; §6 wires that back in)
CSAT = {"chat": 4.1, "email": 3.6, "phone": 4.4, "social": 3.2}
TICKETS = [{"id": 1, "channel": "email", "status": "open"},
           {"id": 2, "channel": "chat",  "status": "closed"},
           {"id": 3, "channel": "email", "status": "open"}]

@dataclass
class Specialist:
    name: str
    role: str                                   # system prompt in a real agent
    act: Callable[[str, dict], dict]            # (task, blackboard) -> result

    def __call__(self, task, blackboard):
        out = self.act(task, blackboard)
        print(f"   🤖 {self.name}: {out}")
        return out

# Analyst: turns a question into a metric using the toolbox
def analyst_act(task, bb):
    t = task.lower()
    if "csat" in t or "satisfaction" in t:
        ch = next((c for c in CSAT if c in t), "email")
        return {"metric": "csat", "channel": ch, "value": CSAT[ch]}
    if "ticket" in t or "open" in t:
        n = sum(x["status"] == "open" for x in TICKETS)
        return {"metric": "open_tickets", "value": n}
    return {"metric": None, "note": "no metric matched"}

analyst = Specialist("analyst", "You compute support metrics from tools.", analyst_act)
print(analyst("What is the CSAT for phone?", {}))

## 3. A second specialist — the writer

The writer never touches data; it turns the analyst's structured result into a customer-ready sentence. Separating "find the number" from "phrase the answer" makes each independently testable.

In [ ]:
def writer_act(task, bb):
    finding = bb.get("finding", {})
    if finding.get("metric") == "csat":
        msg = (f"Our {finding['channel']} channel currently averages "
               f"{finding['value']}/5 in customer satisfaction.")
    elif finding.get("metric") == "open_tickets":
        msg = f"There are {finding['value']} open tickets right now."
    else:
        msg = "I couldn't find a relevant metric for that request."
    return {"reply": msg}

writer = Specialist("writer", "You write concise customer replies.", writer_act)
# writer reads the blackboard the analyst wrote to:
bb = {"finding": analyst("CSAT for chat?", {})}
print(writer("draft a reply", bb))

## 4. The orchestrator

The **orchestrator** owns the plan: it routes the request to the right specialist(s), passes state between them via a **blackboard**, and returns the final result. It is the only component that knows the *workflow*; specialists stay ignorant of each other.

In [ ]:
class Orchestrator:
    def __init__(self, specialists: dict[str, Specialist]):
        self.specialists = specialists

    def run(self, request: str) -> dict:
        bb: dict = {"request": request}            # the shared blackboard
        print(f"📋 orchestrator received: {request!r}")
        # Step 1: analyst finds the metric
        bb["finding"] = self.specialists["analyst"](request, bb)
        # Step 2: writer phrases it
        bb["reply"] = self.specialists["writer"](request, bb)["reply"]
        print(f"✅ final reply: {bb['reply']}")
        return bb

orch = Orchestrator({"analyst": analyst, "writer": writer})
_ = orch.run("What's our phone CSAT and can you phrase it for a customer?")

## 5. Routing & handoff

Not every request needs every specialist. A **router** picks the entry specialist; **handoff** lets one agent pass control to another (e.g. analyst → writer, or escalate → human). Here's a router that sends pure-data questions straight to the analyst and phrasing requests through the full chain.

In [ ]:
def route(request: str) -> str:
    r = request.lower()
    if any(w in r for w in ("reply", "phrase", "customer", "draft", "email back")):
        return "full"          # analyst -> writer
    return "data_only"         # analyst only

def smart_run(orch: Orchestrator, request: str) -> dict:
    mode = route(request); bb = {"request": request, "mode": mode}
    bb["finding"] = orch.specialists["analyst"](request, bb)
    if mode == "full":
        bb["reply"] = orch.specialists["writer"](request, bb)["reply"]
    print(f"   routed as: {mode}")
    return bb

print(smart_run(orch, "How many open tickets?"))            # data_only
print(smart_run(orch, "Draft a customer reply about chat CSAT"))   # full chain

## 6. Capstone — an MCP-backed support-operations assistant

Now the whole module in one system. We stand up a tiny **MCP server** (NB 41), connect a **client**, give the **analyst** its tools *through MCP* (so tools are discovered, not hard-coded), and let the **orchestrator** drive analyst → writer. Swap the offline brains for a real model and this is a shippable support copilot.

In [ ]:
# --- Compact MCP server + client (condensed from NB 41) ---------------------
def rpc(id, method, params=None): return {"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}}

class MiniMCPServer:
    def __init__(self): self.tools = {}
    def add_tool(self, name, desc, fn): self.tools[name] = {"desc": desc, "fn": fn}
    def handle(self, req):
        m, p = req["method"], req.get("params", {})
        if m == "tools/list":
            return {"jsonrpc": "2.0", "id": req["id"],
                    "result": {"tools": [{"name": n, "description": t["desc"]}
                                         for n, t in self.tools.items()]}}
        if m == "tools/call":
            out = self.tools[p["name"]]["fn"](**p.get("arguments", {}))
            return {"jsonrpc": "2.0", "id": req["id"],
                    "result": {"content": [{"type": "text", "text": json.dumps(out)}]}}
        return {"jsonrpc": "2.0", "id": req["id"], "error": {"code": -32601, "message": m}}

class MiniMCPClient:
    def __init__(self, server): self.s = server; self._id = 0
    def _c(self, method, params=None):
        self._id += 1; r = self.s.handle(rpc(self._id, method, params))
        if "error" in r: raise RuntimeError(r["error"]["message"])
        return r["result"]
    def list_tools(self): return self._c("tools/list")["tools"]
    def call_tool(self, name, **a): return json.loads(self._c("tools/call",
                                    {"name": name, "arguments": a})["content"][0]["text"])

# Build the server
ops = MiniMCPServer()
ops.add_tool("get_csat", "CSAT for a channel.",
             lambda channel: {"channel": channel, "csat": CSAT.get(channel.lower())})
ops.add_tool("count_open", "Number of open tickets.",
             lambda: {"open": sum(t["status"] == "open" for t in TICKETS)})
ops_client = MiniMCPClient(ops)
print("server advertises:", [t["name"] for t in ops_client.list_tools()])

In [ ]:
# --- Analyst that uses the MCP client (tools discovered at runtime) ----------
def mcp_analyst_act(task, bb):
    tools = {t["name"] for t in ops_client.list_tools()}
    t = task.lower()
    if "csat" in t and "get_csat" in tools:
        ch = next((c for c in CSAT if c in t), "email")
        r = ops_client.call_tool("get_csat", channel=ch)
        return {"metric": "csat", "channel": r["channel"], "value": r["csat"]}
    if ("ticket" in t or "open" in t) and "count_open" in tools:
        return {"metric": "open_tickets", "value": ops_client.call_tool("count_open")["open"]}
    return {"metric": None}

mcp_analyst = Specialist("analyst", "Computes metrics via MCP tools.", mcp_analyst_act)
capstone = Orchestrator({"analyst": mcp_analyst, "writer": writer})

print("\n=== CAPSTONE RUN ===")
_ = capstone.run("A customer asked about phone support quality — draft a reply.")

## 7. Evaluating a multi-agent system

Test the **system**, not just the parts: feed end-to-end requests and assert the final reply is correct. A few golden cases catch most regressions when you later swap in a real model.

In [ ]:
CASES = [
    ("Draft a reply about phone CSAT", "4.4"),
    ("Draft a reply about chat CSAT",  "4.1"),
    ("Draft a reply about open tickets", "2"),
]
passed = 0
for request, expected in CASES:
    bb = capstone.run(request)
    ok = expected in bb.get("reply", "")
    passed += ok
    print(f"   {'✅' if ok else '❌'} expected {expected!r} in reply\n")
print(f"SYSTEM SCORE: {passed}/{len(CASES)}")

## 8. Going live — the real multi-agent sketch

Offline, each `act()` is deterministic. With a real provider, each specialist is a `react_agent` (NB 39) whose system prompt is its `role`, whose tools come from an MCP client (NB 41), and whose output the orchestrator routes.

In [ ]:
# Reference only — the shape with a real model + MCP:
#
# from llm_providers import AnthropicLLM
# llm = AnthropicLLM(model="claude-haiku-4-5-20251001")
#
# def specialist_act(role, tools_client):
#     def act(task, bb):
#         schemas = tools_client.list_tools()
#         # run the NB 39 react loop with `role` as the system prompt and
#         # tools_client.call_tool as the executor; return the final answer.
#         ...
#     return act
#
# analyst = Specialist("analyst", "Compute metrics via tools.", specialist_act(...))
print("(reference — the offline orchestrator above is the runnable version)")

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a third specialist

Add a `critic` specialist that flags a reply if it contains no number, and call it on a draft.

In [ ]:
def critic_act(task, bb):
    reply = bb.get("reply", "")
    return {"ok": bool(re.search(r"\d", reply)), "reply": reply}

critic = Specialist("critic", "Checks replies are grounded.", critic_act)
bb = capstone.run("Draft a reply about chat CSAT")
print(critic("review", bb))

### Exercise 2 — ⭐⭐ A new tool, zero analyst changes

Add a `worst_channel` tool to the MCP server and confirm `list_tools()` shows it without touching the client.

In [ ]:
ops.add_tool("worst_channel", "Channel with the lowest CSAT.",
             lambda: {"channel": min(CSAT, key=CSAT.get), "csat": min(CSAT.values())})
print([t["name"] for t in ops_client.list_tools()])
print(ops_client.call_tool("worst_channel"))

### Exercise 3 — ⭐⭐ Count specialist calls

Wrap `Specialist.__call__` usage to count how many times each specialist runs during a capstone request (a basic cost metric).

In [ ]:
calls = {}
def counting(spec):
    def wrapped(task, bb):
        calls[spec.name] = calls.get(spec.name, 0) + 1
        return spec.act(task, bb)
    return Specialist(spec.name, spec.role, wrapped)

c_orch = Orchestrator({"analyst": counting(mcp_analyst), "writer": counting(writer)})
c_orch.run("Draft a reply about phone CSAT")
print("specialist calls:", calls)

### Exercise 4 — ⭐⭐ Debug me 🐞

This loop is meant to print each case's expected value, but it errors. Read it, then fix (next cell).

In [ ]:
# 🐞 BUG (INTENTIONALLY ERRORS): unpacking a 2-tuple into 3 names.
for request, expected, mode in CASES:
    print(request, "->", expected)

In [ ]:
# ✅ Fix: CASES holds (request, expected) pairs — unpack two.
for request, expected in CASES:
    print(request, "->", expected)

## 🧠 Stretch exercises

### Stretch A — ⭐⭐⭐ Parallel specialists

When sub-tasks are independent (e.g. fetch CSAT *and* ticket count), run two analyst calls in parallel and merge the findings.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def parallel_findings(requests):
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(lambda r: mcp_analyst_act(r, {}), requests))

print(parallel_findings(["phone csat", "open tickets", "chat csat"]))

### Stretch B — ⭐⭐⭐ Orchestrator with a budget

Give the orchestrator a `max_specialist_calls` budget and stop once it's hit — the multi-agent version of NB 39's step budget.

In [ ]:
def budgeted_run(specs, request, budget=2):
    bb, used = {"request": request}, 0
    for name in ("analyst", "writer"):
        if used >= budget: bb["stopped"] = f"budget {budget} hit"; break
        out = specs[name].act(request, bb); used += 1
        bb["finding" if name == "analyst" else "reply"] = out if name == "analyst" else out["reply"]
    return bb

print(budgeted_run({"analyst": mcp_analyst, "writer": writer}, "Draft a reply about chat CSAT", budget=2))
print(budgeted_run({"analyst": mcp_analyst, "writer": writer}, "Draft a reply about chat CSAT", budget=1))

### Stretch C — ⭐⭐⭐ Escalation handoff

If the analyst finds no metric, hand off to a `human` specialist that returns an escalation ticket instead of a guessed answer.

In [ ]:
def human_act(task, bb):
    return {"escalated": True, "ticket": f"ESC-{len(TICKETS)+1}", "reason": "no metric matched"}
human = Specialist("human", "Handles escalations.", human_act)

def run_with_escalation(specs, request):
    bb = {"request": request}
    bb["finding"] = specs["analyst"](request, bb)
    if bb["finding"].get("metric") is None:
        bb["handoff"] = specs["human"](request, bb)
    else:
        bb["reply"] = specs["writer"](request, bb)["reply"]
    return bb

specs = {"analyst": mcp_analyst, "writer": writer, "human": human}
print(run_with_escalation(specs, "What is the meaning of life?"))   # escalates

### Stretch D — ⭐⭐⭐ A blackboard log

Record every write to the blackboard with the specialist that made it, so you can replay how a decision was reached (multi-agent observability).

In [ ]:
class LoggedBlackboard(dict):
    def __init__(self): super().__init__(); self.history = []
    def write(self, who, key, value):
        self[key] = value; self.history.append({"by": who, "key": key})

lb = LoggedBlackboard()
lb.write("analyst", "finding", mcp_analyst_act("phone csat", {}))
lb.write("writer", "reply", writer_act("draft", lb)["reply"])
print("final reply:", lb["reply"])
print("audit trail:", lb.history)

## 🎁 Bonus mini-project — the full assistant as one function

Wrap the capstone into `support_assistant(request)` that routes, runs the right specialists through MCP, critiques the reply, and returns a structured result. This is the module's finished artifact — and the template for any real agentic product you build next.

In [ ]:
def support_assistant(request: str) -> dict:
    bb = {"request": request, "mode": route(request)}
    bb["finding"] = mcp_analyst_act(request, bb)
    if bb["mode"] == "full" and bb["finding"].get("metric"):
        bb["reply"] = writer_act(request, bb)["reply"]
        bb["grounded"] = bool(re.search(r"\d", bb["reply"]))
    elif bb["finding"].get("metric") is None:
        bb["reply"] = human_act(request, bb); bb["grounded"] = False
    else:
        bb["reply"] = bb["finding"]; bb["grounded"] = True
    return bb

print(json.dumps(support_assistant("Draft a customer reply about phone CSAT"), indent=2))

## 🧠 Key takeaways

1. **Reach for multiple agents only when sub-tasks differ** in tools, instructions, owners, or need independent review — otherwise one agent wins.
2. The **orchestrator–worker** pattern keeps the workflow in one place; specialists stay narrow and testable.
3. **Route** to the right specialist and **hand off** (including to a human) instead of guessing.
4. Share state through a **blackboard**; log writes for observability.
5. **Evaluate the whole system** with golden end-to-end cases, not just unit pieces.
6. The capstone shows the module's stack end-to-end: **agents (39) + robust tools (40) + MCP (41)** → a real support copilot.
7. Every offline brain here swaps for a real model with **no change to the orchestration.**

## ✅ Self-assessment

- [ ] State when a multi-agent design is worth its cost
- [ ] Build an orchestrator that routes between specialist agents
- [ ] Pass state via a blackboard and add an escalation handoff
- [ ] Give the analyst its tools through an MCP client
- [ ] Evaluate the system with end-to-end golden cases
- [ ] Explain how to swap the offline brains for a real model

## 🚀 Where to go next

You've completed **Module 11 — Agents, Tools & MCP**. From here:

- **Ship it:** turn the capstone server into a real **`mcp` FastMCP** server (NB 41 §9) and register it with **Claude Desktop** or **Claude Code**.
- **Go deeper:** revisit **Module 9 (Building AI POCs)** and **Module 5 (AI Engineering)** with agent eyes.
- **Build something:** point an orchestrator at *your* data via an MCP server — that's a portfolio project.